# 03 — Explainability (SHAP)
## NetGuard AI: ML-Powered Network Intrusion Detection

This notebook demonstrates SHAP-based model explainability:
- Global feature importance
- Per-prediction explanations
- Why the model classified specific traffic as an attack

In [ ]:
import sys
sys.path.insert(0, '..')

import os, joblib
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from netguard.preprocessing.loader import load_nsl_kdd
from netguard.preprocessing.features import prepare_dataset
from netguard.explainability.shap_explainer import SHAPExplainer

os.makedirs('../docs/figures', exist_ok=True)
print('Ready')

## 1. Load Model and Data

In [ ]:
rf_model = joblib.load('../models/rf_model.pkl')
xgb_model = joblib.load('../models/xgb_model.pkl')

test_df = load_nsl_kdd('test')
X_test, y_test, scaler, features = prepare_dataset(test_df, top_k_features=0)

# Use subset for SHAP (faster)
X_sample = X_test.iloc[:500]
y_sample = y_test.iloc[:500]

print(f'Sample: {X_sample.shape}')
print(f'Features: {len(features)}')

## 2. Global SHAP — Random Forest

In [ ]:
rf_explainer = SHAPExplainer(rf_model, model_type='tree')
rf_explainer.plot_summary(X_sample, save_path='../docs/figures/shap_summary_rf.png', max_display=15)

top_rf = rf_explainer.get_top_features(X_sample, top_k=10)
print('Top 10 SHAP features (RF):', top_rf)

## 3. Global SHAP — XGBoost

In [ ]:
xgb_explainer = SHAPExplainer(xgb_model, model_type='tree')
xgb_explainer.plot_summary(X_sample, save_path='../docs/figures/shap_summary_xgb.png', max_display=15)

top_xgb = xgb_explainer.get_top_features(X_sample, top_k=10)
print('Top 10 SHAP features (XGBoost):', top_xgb)

## 4. Single Prediction Explanation

In [ ]:
# Find an attack sample
attack_indices = y_sample[y_sample == 1].index[:5]
normal_indices = y_sample[y_sample == 0].index[:5]

print('=== ATTACK SAMPLE ===')
if len(attack_indices) > 0:
    idx = attack_indices[0]
    pos = X_sample.index.get_loc(idx)
    pred = rf_model.predict(X_sample.iloc[[pos]])[0]
    print(f'Index: {idx}, Predicted: {"Attack" if pred == 1 else "Normal"}')
    
    contributions = rf_explainer.explain_single(X_sample, index=pos)
    print('\nTop contributing features:')
    for feat, val in list(contributions.items())[:10]:
        direction = 'Attack ↑' if val > 0 else 'Normal ↓'
        print(f'  {feat}: {val:+.4f} ({direction})')
    
    rf_explainer.plot_waterfall(X_sample, index=pos, save_path='../docs/figures/shap_waterfall_attack.png')

In [ ]:
print('=== NORMAL SAMPLE ===')
if len(normal_indices) > 0:
    idx = normal_indices[0]
    pos = X_sample.index.get_loc(idx)
    pred = rf_model.predict(X_sample.iloc[[pos]])[0]
    print(f'Index: {idx}, Predicted: {"Attack" if pred == 1 else "Normal"}')
    
    contributions = rf_explainer.explain_single(X_sample, index=pos)
    print('\nTop contributing features:')
    for feat, val in list(contributions.items())[:10]:
        direction = 'Attack ↑' if val > 0 else 'Normal ↓'
        print(f'  {feat}: {val:+.4f} ({direction})')
    
    rf_explainer.plot_waterfall(X_sample, index=pos, save_path='../docs/figures/shap_waterfall_normal.png')

## 5. Compare SHAP between Models

In [ ]:
print('=== Feature Importance Comparison (SHAP) ===')
comparison = pd.DataFrame({
    'RF': pd.Series({f: 0 for f in top_rf}, name='RF'),
    'XGBoost': pd.Series({f: 0 for f in top_xgb}, name='XGBoost'),
})

for i, f in enumerate(top_rf):
    comparison.loc[f, 'RF'] = len(top_rf) - i
for i, f in enumerate(top_xgb):
    comparison.loc[f, 'XGBoost'] = len(top_xgb) - i

comparison = comparison.fillna(0).sort_values('RF', ascending=False)
print(comparison)

common = set(top_rf) & set(top_xgb)
print(f'\nFeatures in common (both top-10): {common}')

## Summary

SHAP analysis reveals:
- **Most important features** for attack detection across both models
- **Per-prediction explanations** showing why each connection was classified
- Models agree on key features but may differ in secondary importance

This explainability is crucial for security analysts who need to understand and trust the detection system.